# Deutsche Bahn Delay Predictor — Exploratory Data Analysis

**Dataset:** `data/training_data.csv`  
**Source:** HuggingFace `piebro/deutsche-bahn-data` + Project-1 PostgreSQL  
**Goal:** Understand delay patterns before feature engineering and model training.

---
### Table of Contents
1. [Setup & Load](#1-setup)
2. [Dataset Overview](#2-overview)
3. [Delay Distribution](#3-delay-dist)
4. [Delays by Train Type](#4-train-type)
5. [Delays by Hour of Day](#5-hour)
6. [Delays by Day of Week](#6-dow)
7. [Delays by Station (Top 20)](#7-station)
8. [Cancellations](#8-cancellations)
9. [Correlation Analysis](#9-correlation)
10. [Missing Values & Outliers](#10-quality)
11. [Key Takeaways](#11-takeaways)

## 1 — Setup & Load <a id='1-setup'></a>

In [3]:
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# ── Style ────────────────────────────────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.titlesize': 13,
    'axes.titleweight': 'bold',
})
DB_RED   = '#CC0000'   # Deutsche Bahn brand red
DB_GREY  = '#646973'
DB_LIGHT = '#F0F0F0'

# ── Paths (works from notebooks/ or project root) ────────────────────────────
_here     = Path().resolve()
ROOT      = _here.parent if _here.name == 'notebooks' else _here
DATA_PATH = ROOT / 'data' / 'training_data.csv'
FIG_DIR   = ROOT / 'data'

# ── Load ─────────────────────────────────────────────────────────────────────
df = pd.read_csv(DATA_PATH, parse_dates=['planned_departure'])
print(f'Loaded {len(df):,} rows × {df.shape[1]} columns')
df.head(3)

Error: No connection selected.

## 2 — Dataset Overview <a id='2-overview'></a>

In [4]:
print('=== Shape ===')
print(f'  Rows: {len(df):,}   Columns: {df.shape[1]}')

print('\n=== Date range ===')
print(f'  {df["planned_departure"].min().date()}  →  {df["planned_departure"].max().date()}')

print('\n=== Data types ===')
print(df.dtypes.to_string())

print('\n=== Target class balance ===')
vc = df['is_delayed'].value_counts()
print(f'  Not delayed:  {vc[False]:>7,}  ({vc[False]/len(df)*100:.1f}%)')
print(f'  Delayed >5min:{vc[True]:>7,}  ({vc[True]/len(df)*100:.1f}%)')

print('\n=== Delay statistics (minutes) ===')
print(df['delay_minutes'].describe().round(2).to_string())

Error: No connection selected.

In [5]:
# Class balance pie
fig, ax = plt.subplots(figsize=(5, 5))
labels = ['On time / early\n(≤ 5 min)', 'Delayed\n(> 5 min)']
sizes  = [vc[False], vc[True]]
colors = [sns.color_palette('muted')[0], DB_RED]
wedges, texts, autotexts = ax.pie(
    sizes, labels=labels, colors=colors, autopct='%1.1f%%',
    startangle=90, wedgeprops=dict(edgecolor='white', linewidth=2)
)
for at in autotexts:
    at.set_fontsize(12)
ax.set_title('Class Balance — is_delayed target', pad=14)
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig_class_balance.png', bbox_inches='tight')
plt.show()

Error: No connection selected.

## 3 — Delay Distribution <a id='3-delay-dist'></a>

In [6]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# 3a  Full distribution (clipped for readability)
clip = df['delay_minutes'].clip(-20, 60)
axes[0].hist(clip, bins=80, color=DB_RED, edgecolor='white', linewidth=0.3)
axes[0].axvline(5,  color='black', linestyle='--', linewidth=1.2, label='5-min threshold')
axes[0].axvline(df['delay_minutes'].median(), color=DB_GREY,
                linestyle=':', linewidth=1.4, label=f'Median = {df["delay_minutes"].median():.0f}')
axes[0].set_title('Delay Distribution (clipped −20 to +60)')
axes[0].set_xlabel('Delay (minutes)')
axes[0].set_ylabel('Count')
axes[0].legend(fontsize=9)

# 3b  CDF
sorted_d = np.sort(df['delay_minutes'].values)
cdf = np.arange(1, len(sorted_d)+1) / len(sorted_d)
axes[1].plot(sorted_d.clip(-10, 60), cdf, color=DB_RED, linewidth=1.5)
axes[1].axvline(5, color='black', linestyle='--', linewidth=1.2, label='5-min cutoff')
axes[1].axhline(np.mean(df['delay_minutes'] <= 5), color=DB_GREY,
                linestyle=':', linewidth=1.2)
axes[1].set_title('CDF of Delay (clipped)')
axes[1].set_xlabel('Delay (minutes)')
axes[1].set_ylabel('Cumulative probability')
axes[1].legend(fontsize=9)

# 3c  Box plot by delayed flag
delayed_clip = df.copy()
delayed_clip['delay_clip'] = delayed_clip['delay_minutes'].clip(-10, 60)
delayed_clip['delay_label'] = delayed_clip['is_delayed'].map({False: 'Not delayed', True: 'Delayed > 5 min'})
sns.boxplot(
    data=delayed_clip, x='delay_label', y='delay_clip',
    order=['Not delayed', 'Delayed > 5 min'],
    palette={'Not delayed': sns.color_palette('muted')[0], 'Delayed > 5 min': DB_RED},
    ax=axes[2], width=0.4
)
axes[2].set_title('Box Plot by Delay Class')
axes[2].set_xlabel('')
axes[2].set_ylabel('Delay (minutes, clipped)')

plt.suptitle('Delay Distribution Analysis', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig_delay_distribution.png', bbox_inches='tight')
plt.show()

# Percentile table
pcts = [50, 75, 90, 95, 99]
print('Delay percentiles:')
for p in pcts:
    print(f'  p{p:>2}:  {np.percentile(df["delay_minutes"], p):.0f} min')

Error: No connection selected.

## 4 — Delays by Train Type <a id='4-train-type'></a>

In [7]:
train_stats = (
    df.groupby('train_type')
    .agg(
        total       = ('delay_minutes', 'count'),
        avg_delay   = ('delay_minutes', 'mean'),
        med_delay   = ('delay_minutes', 'median'),
        delay_rate  = ('is_delayed',    'mean'),
        cancel_rate = ('is_cancelled',  'mean'),
    )
    .sort_values('avg_delay', ascending=False)
    .round(2)
)
train_stats['delay_rate_pct']  = (train_stats['delay_rate']  * 100).round(1)
train_stats['cancel_rate_pct'] = (train_stats['cancel_rate'] * 100).round(1)
print(train_stats[['total','avg_delay','med_delay','delay_rate_pct','cancel_rate_pct']].to_string())

Error: No connection selected.

In [8]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
order = train_stats.sort_values('avg_delay', ascending=False).index.tolist()

# 4a  Average delay
sns.barplot(
    x=train_stats.loc[order, 'avg_delay'], y=order,
    palette='Reds_r', ax=axes[0]
)
axes[0].set_title('Average Delay by Train Type')
axes[0].set_xlabel('Avg delay (min)')
axes[0].set_ylabel('')

# 4b  Delay rate %
sns.barplot(
    x=train_stats.loc[order, 'delay_rate_pct'], y=order,
    palette='Oranges_r', ax=axes[1]
)
axes[1].set_title('Delay Rate > 5 min (%)')
axes[1].set_xlabel('% of departures delayed')
axes[1].set_ylabel('')

# 4c  Volume
sns.barplot(
    x=train_stats.loc[order, 'total'], y=order,
    palette='Blues_r', ax=axes[2]
)
axes[2].set_title('Volume (# departures)')
axes[2].set_xlabel('Count')
axes[2].set_ylabel('')
axes[2].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}k'))

plt.suptitle('Delay Analysis by Train Type', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig_delay_by_train_type.png', bbox_inches='tight')
plt.show()

Error: No connection selected.

## 5 — Delays by Hour of Day <a id='5-hour'></a>

In [9]:
hour_stats = (
    df.groupby('hour')
    .agg(
        avg_delay  = ('delay_minutes', 'mean'),
        delay_rate = ('is_delayed',    'mean'),
        volume     = ('delay_minutes', 'count'),
    )
    .reset_index()
)

# Rush-hour bands
MORNING_RUSH = [7, 8, 9]
EVENING_RUSH = [17, 18, 19]

fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True)

# Top — avg delay line
axes[0].plot(hour_stats['hour'], hour_stats['avg_delay'],
             color=DB_RED, linewidth=2.5, marker='o', markersize=5)
for h in MORNING_RUSH + EVENING_RUSH:
    axes[0].axvspan(h - 0.5, h + 0.5, alpha=0.12, color='gold')
axes[0].set_ylabel('Avg delay (min)')
axes[0].set_title('Average Delay by Hour of Day')
axes[0].text(8, hour_stats['avg_delay'].max() * 0.95, 'Morning rush',
             fontsize=8, color='goldenrod')
axes[0].text(17.5, hour_stats['avg_delay'].max() * 0.95, 'Evening rush',
             fontsize=8, color='goldenrod')

# Bottom — volume bars coloured by delay rate
bar_colors = plt.cm.Reds(hour_stats['delay_rate'] / hour_stats['delay_rate'].max())
axes[1].bar(hour_stats['hour'], hour_stats['volume'], color=bar_colors, edgecolor='white')
axes[1].set_ylabel('# Departures')
axes[1].set_xlabel('Hour of day')
axes[1].set_title('Departure Volume by Hour (colour = delay rate)')
axes[1].set_xticks(range(0, 24))

plt.tight_layout()
plt.savefig(FIG_DIR / 'fig_delay_by_hour.png', bbox_inches='tight')
plt.show()

Error: No connection selected.

## 6 — Delays by Day of Week <a id='6-dow'></a>

In [10]:
DOW_LABELS = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']

dow_stats = (
    df.groupby('day_of_week')
    .agg(
        avg_delay  = ('delay_minutes', 'mean'),
        delay_rate = ('is_delayed',    'mean'),
        volume     = ('delay_minutes', 'count'),
    )
    .reindex(range(7))
    .reset_index()
)
dow_stats['label'] = [DOW_LABELS[i] for i in dow_stats['day_of_week']]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

palette = [DB_RED if i >= 5 else sns.color_palette('muted')[0] for i in range(7)]

# Avg delay
axes[0].bar(dow_stats['label'], dow_stats['avg_delay'], color=palette, edgecolor='white')
axes[0].set_title('Average Delay by Day of Week')
axes[0].set_ylabel('Avg delay (min)')
axes[0].set_xticks(range(7))
axes[0].set_xticklabels(dow_stats['label'], rotation=30, ha='right')
axes[0].axvspan(4.5, 6.5, alpha=0.08, color=DB_RED, label='Weekend')
axes[0].legend(fontsize=9)

# Delay rate
axes[1].bar(dow_stats['label'], dow_stats['delay_rate'] * 100, color=palette, edgecolor='white')
axes[1].set_title('Delay Rate > 5 min by Day of Week')
axes[1].set_ylabel('% delayed')
axes[1].set_xticks(range(7))
axes[1].set_xticklabels(dow_stats['label'], rotation=30, ha='right')
axes[1].axhline(dow_stats['delay_rate'].mean() * 100, color='black',
                linestyle='--', linewidth=1.2, label='Overall avg')
axes[1].legend(fontsize=9)

plt.suptitle('Delay Analysis by Day of Week  (red = weekend)', fontsize=13,
             fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig_delay_by_dow.png', bbox_inches='tight')
plt.show()

print(dow_stats[['label','avg_delay','delay_rate','volume']].round(3).to_string(index=False))

Error: No connection selected.

## 7 — Delays by Station (Top 20) <a id='7-station'></a>

In [11]:
# Require at least 100 departures to appear
station_stats = (
    df.groupby('station_name')
    .agg(
        total      = ('delay_minutes', 'count'),
        avg_delay  = ('delay_minutes', 'mean'),
        delay_rate = ('is_delayed',    'mean'),
    )
    .query('total >= 100')
)

top20_avg    = station_stats.nlargest(20, 'avg_delay')
top20_rate   = station_stats.nlargest(20, 'delay_rate')
top20_volume = station_stats.nlargest(20, 'total')

print(f'Stations with ≥100 departures: {len(station_stats):,}')

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Left — top 20 by avg delay
sns.barplot(
    data=top20_avg.reset_index(),
    x='avg_delay', y='station_name',
    palette='Reds_r', ax=axes[0]
)
axes[0].set_title('Top 20 Stations — Highest Avg Delay')
axes[0].set_xlabel('Avg delay (min)')
axes[0].set_ylabel('')

# Right — top 20 by delay rate
sns.barplot(
    data=top20_rate.reset_index(),
    x='delay_rate', y='station_name',
    palette='Oranges_r', ax=axes[1]
)
axes[1].set_title('Top 20 Stations — Highest Delay Rate (> 5 min)')
axes[1].set_xlabel('Delay rate')
axes[1].set_ylabel('')
axes[1].xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))

plt.suptitle('Station-level Delay Analysis', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig_delay_by_station.png', bbox_inches='tight')
plt.show()

Error: No connection selected.

In [12]:
# Heatmap: avg delay — top 15 stations × train type
top15_stations = station_stats.nlargest(15, 'total').index.tolist()
pivot = (
    df[df['station_name'].isin(top15_stations)]
    .groupby(['station_name', 'train_type'])['delay_minutes']
    .mean()
    .unstack(fill_value=np.nan)
)

fig, ax = plt.subplots(figsize=(12, 6))
sns.heatmap(
    pivot, annot=True, fmt='.1f', cmap='YlOrRd',
    linewidths=0.4, ax=ax, cbar_kws={'label': 'Avg delay (min)'}
)
ax.set_title('Avg Delay Heatmap — Top 15 Stations × Train Type', pad=12)
ax.set_xlabel('Train type')
ax.set_ylabel('')
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig_station_traintype_heatmap.png', bbox_inches='tight')
plt.show()

Error: No connection selected.

## 8 — Cancellations <a id='8-cancellations'></a>

In [13]:
cancel_by_type = (
    df.groupby('train_type')['is_cancelled']
    .agg(['sum', 'mean'])
    .rename(columns={'sum': 'cancelled_count', 'mean': 'cancel_rate'})
    .sort_values('cancel_rate', ascending=False)
)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# By train type
sns.barplot(
    x=cancel_by_type['cancel_rate'] * 100, y=cancel_by_type.index,
    palette='Blues_r', ax=axes[0]
)
axes[0].set_title('Cancellation Rate by Train Type')
axes[0].set_xlabel('% cancelled')
axes[0].set_ylabel('')

# By hour
cancel_hour = df.groupby('hour')['is_cancelled'].mean() * 100
axes[1].bar(cancel_hour.index, cancel_hour.values, color=DB_RED, edgecolor='white')
axes[1].set_title('Cancellation Rate by Hour of Day')
axes[1].set_xlabel('Hour')
axes[1].set_ylabel('% cancelled')
axes[1].set_xticks(range(0, 24))

plt.tight_layout()
plt.savefig(FIG_DIR / 'fig_cancellations.png', bbox_inches='tight')
plt.show()

print(f"Overall cancellation rate: {df['is_cancelled'].mean()*100:.2f}%")
print(cancel_by_type.round(3).to_string())

Error: No connection selected.

## 9 — Correlation Analysis <a id='9-correlation'></a>

In [14]:
from sklearn.preprocessing import LabelEncoder

# Encode categoricals for correlation
corr_df = df.copy()
le = LabelEncoder()
corr_df['train_type_enc'] = le.fit_transform(corr_df['train_type'])
corr_df['station_enc']    = le.fit_transform(corr_df['station_name'])
corr_df['is_delayed_int'] = corr_df['is_delayed'].astype(int)
corr_df['is_cancelled_int'] = corr_df['is_cancelled'].astype(int)

corr_cols = [
    'hour', 'day_of_week', 'train_type_enc', 'station_enc',
    'is_cancelled_int', 'delay_minutes', 'is_delayed_int'
]
corr_matrix = corr_df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(8, 6))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.2f',
    cmap='RdBu_r', center=0, vmin=-1, vmax=1,
    linewidths=0.5, ax=ax,
    xticklabels=['Hour','Day','Train type','Station','Cancelled','Delay(min)','Is delayed'],
    yticklabels=['Hour','Day','Train type','Station','Cancelled','Delay(min)','Is delayed'],
)
ax.set_title('Feature Correlation Matrix', pad=12)
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig_correlation.png', bbox_inches='tight')
plt.show()

Error: No connection selected.

In [15]:
# Delay vs hour scatter with regression line
hour_mean = df.groupby('hour')['delay_minutes'].mean()
hour_rate = df.groupby('hour')['is_delayed'].mean() * 100

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].scatter(hour_mean.index, hour_mean.values, s=80, color=DB_RED, zorder=3)
z = np.polyfit(hour_mean.index, hour_mean.values, 2)
p = np.poly1d(z)
xp = np.linspace(0, 23, 200)
axes[0].plot(xp, p(xp), color=DB_GREY, linestyle='--', linewidth=1.5)
axes[0].set_title('Avg Delay vs Hour (quadratic fit)')
axes[0].set_xlabel('Hour')
axes[0].set_ylabel('Avg delay (min)')

axes[1].scatter(hour_rate.index, hour_rate.values, s=80, color='darkorange', zorder=3)
axes[1].set_title('Delay Rate (>5 min) vs Hour')
axes[1].set_xlabel('Hour')
axes[1].set_ylabel('% delayed')

plt.tight_layout()
plt.show()

Error: No connection selected.

## 10 — Missing Values & Outliers <a id='10-quality'></a>

In [16]:
print('=== Missing values ===')
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
print(pd.DataFrame({'count': missing, 'pct': missing_pct}).to_string())

print('\n=== Outlier analysis — delay_minutes ===')
q1, q3 = df['delay_minutes'].quantile([0.25, 0.75])
iqr    = q3 - q1
lo, hi = q1 - 3 * iqr, q3 + 3 * iqr
outliers = df[(df['delay_minutes'] < lo) | (df['delay_minutes'] > hi)]
print(f'  IQR = {iqr:.1f}  |  fence: [{lo:.1f}, {hi:.1f}]')
print(f'  Outliers (3×IQR): {len(outliers):,}  ({len(outliers)/len(df)*100:.2f}%)')
print(f'  Max delay: {df["delay_minutes"].max():.0f} min  |  Min (early): {df["delay_minutes"].min():.0f} min')

Error: No connection selected.

In [17]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Violin of delay by train type
plot_df = df.copy()
plot_df['delay_clip'] = plot_df['delay_minutes'].clip(-10, 40)
type_order = df.groupby('train_type')['delay_minutes'].mean().sort_values(ascending=False).index
sns.violinplot(
    data=plot_df, x='train_type', y='delay_clip', order=type_order,
    palette='Reds', ax=axes[0], inner='quartile', cut=0
)
axes[0].set_title('Delay Distribution by Train Type (violin)')
axes[0].set_xlabel('Train type')
axes[0].set_ylabel('Delay (min, clipped)')

# Extreme delays
extreme = df[df['delay_minutes'] > 30]['train_type'].value_counts()
axes[1].bar(extreme.index, extreme.values, color=DB_RED, edgecolor='white')
axes[1].set_title('Extreme Delays (> 30 min) by Train Type')
axes[1].set_xlabel('Train type')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.savefig(FIG_DIR / 'fig_outliers.png', bbox_inches='tight')
plt.show()

Error: No connection selected.

## 11 — Key Takeaways <a id='11-takeaways'></a>

In [18]:
# Auto-generate summary statistics for the takeaways cell
delay_rate_overall = df['is_delayed'].mean() * 100
worst_type         = train_stats['avg_delay'].idxmax()
best_type          = train_stats['avg_delay'].idxmin()
worst_type_avg     = train_stats.loc[worst_type, 'avg_delay']
peak_hour          = hour_stats.loc[hour_stats['avg_delay'].idxmax(), 'hour']
peak_hour_delay    = hour_stats['avg_delay'].max()
best_dow           = DOW_LABELS[dow_stats.loc[dow_stats['avg_delay'].idxmin(), 'day_of_week']]
worst_dow          = DOW_LABELS[dow_stats.loc[dow_stats['avg_delay'].idxmax(), 'day_of_week']]
worst_station_avg  = top20_avg.iloc[0].name

print(f"""
┌─────────────────────────────────────────────────────────────────────┐
│                        KEY TAKEAWAYS                                │
├─────────────────────────────────────────────────────────────────────┤
│  Dataset                                                            │
│  • {len(df):,} departures, {df['station_name'].nunique():,} unique stations, {df['train_type'].nunique()} train types │
│  • Overall delay rate (> 5 min): {delay_rate_overall:.1f}%                       │
│  • Cancellation rate: {df['is_cancelled'].mean()*100:.1f}%                              │
│                                                                     │
│  Train type                                                         │
│  • Worst: {worst_type} ({worst_type_avg:.1f} min avg)                               │
│  • Best:  {best_type} ({train_stats.loc[best_type,'avg_delay']:.1f} min avg)                          │
│  • S-Bahn dominates volume but has moderate delays                 │
│                                                                     │
│  Time patterns                                                      │
│  • Peak delay hour: {int(peak_hour):02d}:00 ({peak_hour_delay:.1f} min avg)                    │
│  • Worst day: {worst_dow:<10}  Best day: {best_dow}                  │
│  • Rush hours (7-9, 17-19) show elevated delays                    │
│                                                                     │
│  Station patterns                                                   │
│  • Highest avg delay station: {worst_station_avg[:35]}  │
│  • Large hubs (Berlin Hbf, München Hbf) have high volume          │
│                                                                     │
│  Feature engineering signals                                        │
│  • hour, day_of_week, is_rush_hour → strong temporal signal        │
│  • train_type → major differentiator (encode as ordinal)           │
│  • station + train_type historical stats → strongest predictors    │
└─────────────────────────────────────────────────────────────────────┘
""")

Error: No connection selected.